# Day 2 — Step 1: 데이터 수집

업비트(국내 암호화폐 거래소)에서 3종목의 가격 데이터를 수집하고,  
하나의 표로 합칩니다.

---

## 암호화폐 기초 지식 (처음이라면 꼭 읽어두세요)

### 우리가 다룰 3종목
| 시세 코드 | 이름 | 특징 |
|-----------|------|------|
| KRW-BTC | 비트코인(Bitcoin) | 세계 최초의 암호화폐. 암호화폐 시장의 기준점 |
| KRW-ETH | 이더리움(Ethereum) | 스마트 계약 기능이 있는 2세대 코인 |
| KRW-SOL | 솔라나(Solana) | 거래 속도가 매우 빠른 코인 |

> `KRW-BTC`에서 **KRW**는 한국 원화(₩)를 뜻해요.  
> 즉, "원화로 거래되는 비트코인"이라는 의미입니다.

### OHLCV 데이터란?
주식/코인 가격 데이터를 표현하는 표준 형식입니다.

| 컬럼 | 영어 | 뜻 |
|------|------|----|
| open | Open | 하루 중 **첫 번째** 거래 가격 (시가) |
| high | High | 하루 중 **가장 높았던** 가격 (고가) |
| low | Low | 하루 중 **가장 낮았던** 가격 (저가) |
| close | Close | 하루 중 **마지막** 거래 가격 (종가) |
| volume | Volume | 하루 동안 거래된 **총 수량** (거래량) |

→ 우리는 주로 **종가(close)** 를 기준으로 분석합니다.  
→ 종가가 그날의 "최종 결과"이기 때문입니다.

### Long Format으로 합치는 이유

처음엔 종목마다 파일을 따로 만들려고 했는데, 알아보니  
**하나의 표에 종목 구분 컬럼(`ticker`)을 추가**하는 방식이 더 편하다고 해서 이렇게 했습니다.

```
Wide Format (넓은 형식) - 처음에 떠올린 방식
date        BTC_close    ETH_close    SOL_close
2026-01-01  89,000,000   3,100,000    200,000

Long Format (긴 형식) - 실제로 쓸 방식
ticker    date        close
KRW-BTC  2026-01-01  89,000,000
KRW-ETH  2026-01-01   3,100,000
KRW-SOL  2026-01-01     200,000
```

> Long Format은 나중에 종목이 늘어나도 코드를 안 바꿔도 됩니다. (행만 늘어남)

---
## 0. 라이브러리 불러오기

파이썬의 기본 기능만으로는 데이터 분석이 힘들기 때문에  
전문가들이 미리 만들어둔 도구(라이브러리)를 가져와서 씁니다.

| 라이브러리 | 비유 | 우리가 쓰는 이유 |
|-----------|------|------------------|
| pyupbit | 배달앱 | 업비트 서버에서 코인 가격을 대신 받아다 줌 |
| pandas | 엑셀 | 데이터를 표(DataFrame) 형태로 쉽게 다룰 수 있게 해줌 |

In [ ]:
import pyupbit          # 업비트 코인 가격 데이터 수집
import pandas as pd     # 데이터를 표(DataFrame)로 다루기
import warnings

# 불필요한 경고 메시지 숨기기
# FutureWarning 같은 경고들이 뜨는데, 학습 단계에서는 일단 무시합니다
warnings.filterwarnings('ignore')

print("라이브러리 불러오기 완료!")

---
## 1. 수집 설정

종목 리스트와 수집 기간을 변수로 따로 빼 놓았습니다.  
나중에 종목을 바꾸거나 기간을 조정할 때 **이 셀만 수정**하면 됩니다.

### 왜 60일치를 받아서 30일만 쓰나요?

다음 단계에서 **5일 이동평균(MA5)** 을 계산할 때  
처음 4일은 데이터가 부족해서 빈값(NaN)이 생깁니다.  
넉넉하게 60일치를 받아두고, 실제로는 최근 30일치만 씁니다.

In [ ]:
# ── 분석할 종목 리스트 ────────────────────────────────────────────
# 나중에 종목을 추가하거나 바꾸고 싶을 때 이 리스트만 수정하면 됩니다
TICKERS = ["KRW-BTC", "KRW-ETH", "KRW-SOL"]

# ── 수집 기간 설정 ────────────────────────────────────────────────
FETCH_DAYS = 60   # API에서 받아올 일수 (여유분 포함)
USE_DAYS   = 30   # 실제 분석에 사용할 최근 일수

print(f"분석 종목 : {TICKERS}")
print(f"수집 기간 : 최근 {FETCH_DAYS}일치 수집 → 최근 {USE_DAYS}일만 사용")

---
## 2. 데이터 수집

### 업비트 API란?

API(Application Programming Interface)란 **두 프로그램이 서로 대화하는 방법**입니다.  
우리가 pyupbit를 통해 "BTC 가격 줘!"라고 요청하면,  
업비트 서버가 데이터를 돌려주는 방식입니다.

```
[내 컴퓨터] ──요청──▶ [업비트 서버] ──데이터──▶ [내 컴퓨터]
```

### 왜 여러 종목을 한 번에 처리하나요?

for 반복문으로 종목 리스트를 순서대로 돌면서 각각 데이터를 받아옵니다.  
받아온 데이터를 리스트에 임시 저장한 뒤, 마지막에 `pd.concat()`으로 한 번에 합칩니다.

In [ ]:
def collect_ohlcv(tickers, fetch_days, use_days):
    """
    여러 종목의 OHLCV 데이터를 수집하고 하나의 DataFrame으로 합칩니다.

    매개변수(Parameter):
        tickers    : 종목 코드 리스트 (예: ["KRW-BTC", "KRW-ETH"])
        fetch_days : API에서 받아올 일수
        use_days   : 실제로 쓸 최근 일수 (fetch_days보다 작거나 같아야 함)

    반환값(Return):
        ticker, date, open, high, low, close, volume 컬럼을 가진 DataFrame
    """
    # 종목별 데이터를 임시로 담아두는 리스트
    # 나중에 pd.concat()으로 한 번에 합칩니다
    all_frames = []

    for ticker in tickers:
        print(f"  [{ticker}] 수집 중...")
        try:
            # pyupbit.get_ohlcv() : 업비트 일봉 데이터 수집 함수
            # count=fetch_days    : 최근 몇 일치를 받을지
            df = pyupbit.get_ohlcv(ticker, count=fetch_days)

            if df is None or df.empty:
                print(f"  [{ticker}] 수집 실패, 건너뜀")
                continue

            # .tail(n)  : 마지막 n행만 가져옴 = 가장 최근 n일
            # .copy()   : 원본을 건드리지 않도록 복사본 사용
            df = df.tail(use_days).copy()

            # 날짜(index)를 'date' 컬럼으로 추가
            # .index.date : datetime 인덱스에서 날짜 부분만 뽑기 (시간 제거)
            df['date'] = pd.to_datetime(df.index.date)

            # 종목 코드 컬럼 추가 → 합쳤을 때 어느 종목인지 구분할 수 있음
            df['ticker'] = ticker

            # reset_index(drop=True) : 행 번호를 0, 1, 2... 로 새로 매기기
            # drop=True : 기존 날짜 인덱스는 이미 'date' 컬럼으로 저장했으니 버림
            df = df.reset_index(drop=True)

            all_frames.append(df)
            print(f"  [{ticker}] 완료 ({len(df)}일치)")

        except Exception as e:
            # 예상치 못한 오류가 생겼을 때 해당 종목만 건너뛰고 계속 진행
            print(f"  [{ticker}] 오류 발생: {e}, 건너뜀")
            continue

    # ── 여러 DataFrame을 세로로 합치기 ──────────────────────────────
    # pd.concat()        : 여러 DataFrame을 이어 붙이는 함수
    # axis=0             : 세로 방향으로 쌓기 (행이 늘어남)
    # ignore_index=True  : 합친 후 인덱스를 0, 1, 2... 로 새로 매기기
    combined_df = pd.concat(all_frames, axis=0, ignore_index=True)
    return combined_df

In [ ]:
# ── 데이터 수집 실행 ───────────────────────────────────────────────
print("=== 데이터 수집 시작 ===")
df_raw = collect_ohlcv(TICKERS, FETCH_DAYS, USE_DAYS)
print(f"\n=== 수집 완료: 총 {len(df_raw)}행 ===")

In [ ]:
# ── 필요한 컬럼만 선택하고 정렬 ───────────────────────────────────
# 수집된 데이터에서 분석에 필요한 컬럼만 뽑습니다
columns_needed = ['ticker', 'date', 'open', 'high', 'low', 'close', 'volume']
df_raw = df_raw[columns_needed].copy()

# 종목별로, 날짜 오름차순으로 정렬
# by=['ticker', 'date'] : 먼저 종목 코드 순으로, 같은 종목 안에서는 날짜 순으로
df_raw = df_raw.sort_values(by=['ticker', 'date'], ascending=True)
df_raw = df_raw.reset_index(drop=True)

print("[수집된 데이터 미리보기 (상위 5행)]")
print(df_raw.head())
print()

# .info() : 컬럼명, 데이터 개수, 자료형(dtype)을 한눈에 확인
print("[컬럼 정보]")
df_raw.info()

In [ ]:
# ── 다음 단계(02_preprocessing)로 전달하기 위해 CSV로 저장 ────────
# index=False : 행 번호(인덱스)는 저장하지 않음
df_raw.to_csv('ohlcv_raw.csv', index=False)
print("ohlcv_raw.csv 저장 완료 → 02_preprocessing.ipynb에서 이어서 작업합니다")